# 06 · Integración: API + worker independientes (E.6 pasos 1-11)

Ejecutar con **Kernel limpio → Run All**, en el orden documentado en `docs/NOTEBOOKS.md`.

Arranca **API y worker como procesos separados**, y recorre el flujo solo por HTTP. El kernel no hace procesamiento pesado.

In [ ]:
import sys; sys.path.insert(0, '.')
from _common import *
import requests, shutil
W = workdir(); web = W/'web'; data = W/'int_data'
shutil.rmtree(data, ignore_errors=True)
srv = serve(web, 8911); base = 'http://127.0.0.1:8911'
write_feed(web, base, 1); write_feed(web, base, 2)
CONFIG = {'backend_factory': 'tests.fakes:make_backends', 'block_seconds': 60, 'port': 8912}
data.mkdir(); (data/'config.json').write_text(json.dumps(CONFIG))
(W/'truth.json').write_text(json.dumps([[0,8,'strong'],[39.5,59.5,'strong'],[92.5,107.5,'weak']]))
env = dict(os.environ, PODCLEANY_FAKE_TRUTH=str(W/'truth.json'), PYTHONPATH=str(ROOT))
procs = [subprocess.Popen([sys.executable, '-m', 'podcleany', k, '--data-dir', str(data)], env=env, cwd=ROOT) for k in ('api', 'worker')]
API = 'http://127.0.0.1:8912'
for _ in range(50):
    try:
        requests.get(API + '/health', timeout=1); break
    except Exception: time.sleep(0.3)
print(requests.get(API + '/health').json())

In [ ]:
r = requests.post(API + '/episodes', json={'url': base + '/feed1.xml'}); print(r.status_code, r.json())
jid = r.json()['job_id']
with requests.get(f'{API}/jobs/{jid}/events', stream=True) as s:   # SSE
    for line in s.iter_lines(decode_unicode=True):
        if line.startswith('data:'): print(line[:120])
ep = requests.get(f"{API}/episodes/{requests.get(f'{API}/jobs/{jid}').json()['episode_id']}").json()
for s in ep['segments']: print(s['kind'], round(s['start'],1), round(s['end'],1), s['decision'])

In [ ]:
# retroalimentación: conservar la intro, protegerla y confirmar los anuncios (aprende huellas)
segs = ep['segments']
intro = min(segs, key=lambda s: s['start'])
requests.patch(f"{API}/segments/{intro['id']}", json={'decision': 'keep'})
print(requests.post(f"{API}/episodes/{ep['id']}/protect", json={'start': 0, 'end': 8, 'label': 'intro'}).json())
for s in segs:
    if s is not intro: print(requests.patch(f"{API}/segments/{s['id']}", json={'decision': 'remove'}).json())
print(requests.post(f"{API}/episodes/{ep['id']}/render").json())
time.sleep(4)
ep = requests.get(f"{API}/episodes/{ep['id']}").json(); print(ep['clean'])
r = requests.get(f"{API}/episodes/{ep['id']}/audio/clean", headers={'Range': 'bytes=0-99'}); print(r.status_code, r.headers['Content-Range'])

In [ ]:
# episodio 2: el anuncio A se reconoce por huella (el guion simulado solo marca la intro como sospechosa)
(W/'truth.json').write_text(json.dumps([[0,8,'strong']]))
r = requests.post(API + '/episodes', json={'url': base + '/feed2.xml'}).json(); time.sleep(6)
j = requests.get(f"{API}/jobs/{r['job_id']}").json(); ep2 = requests.get(f"{API}/episodes/{j['episode_id']}").json()
print([(s['source'], round(s['start'],1), round(s['end'],1)) for s in ep2['segments']])

In [ ]:
for p in procs: p.terminate()
srv.shutdown()